# Lab qc_01 — Build a statevector simulator

**Skills:** `qc.tensor`, `qc.multi_gates` · **Prereq (paper):** tensor products & multi-qubit states · ~90 min · CPU only

### Goals
1. Represent an $n$-qubit state as a vector in $\mathbb{C}^{2^n}$ and apply 1- and 2-qubit gates **without** building $2^n\times 2^n$ matrices (reshape + `np.tensordot`).
2. Sample measurement outcomes with the Born rule.
3. Match `qiskit.quantum_info.Statevector` exactly — which forces you to understand **Qiskit's little-endian qubit order**.

### Theory recap
A gate $U$ on qubit $q$ of an $n$-qubit register is $I\otimes\cdots\otimes U\otimes\cdots\otimes I$. Building that Kronecker product costs $O(4^n)$ memory; instead reshape
$\lvert\psi\rangle$ into a tensor $\psi_{b_{n-1}\ldots b_1 b_0}$ with one axis of size 2 per qubit and contract $U$ with **one** axis:
$$\psi'_{\ldots b_q' \ldots} = \sum_{b_q} U_{b_q' b_q}\,\psi_{\ldots b_q \ldots}.$$
A 2-qubit gate is a $(2,2,2,2)$ tensor $U_{a'b',ab}$ contracted with two axes.

**Little-endian (Qiskit).** Qiskit stores the amplitude of the basis state with bits $q_{n-1}\ldots q_1 q_0$ at index $\sum_q q_q 2^q$: qubit 0 is the **least significant / rightmost** bit, so the statevector is $\lvert q_{n-1}\rangle\otimes\cdots\otimes\lvert q_0\rangle$. Textbooks (N&C) write qubit 1 on the *left*. After `np.reshape(psi, [2]*n)`, **axis 0 is qubit $n-1$ and axis $n-1$ is qubit 0**.

Example: `X` on qubit 0 of 2 qubits gives index 1, printed by Qiskit as `'01'`.

**Measurement.** $P(x) = |\langle x\vert\psi\rangle|^2$; sample indices with these weights.

### Read
- de Wolf, *Quantum Computing: Lecture Notes*, Ch. 1–2 — https://arxiv.org/abs/1907.09415
- IBM Quantum Learning, *Basics of Quantum Information* (multiple systems) — https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information
- Qiskit source (`qiskit/quantum_info/states/statevector.py`) — https://github.com/Qiskit/qiskit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

np.set_printoptions(precision=4, suppress=True)

## 1. Gates (given)
Two-qubit matrices are written in the textbook basis $\lvert a\, b\rangle$ where $a$ is the **first** qubit argument (e.g. the control).

In [ ]:
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S = np.diag([1, 1j]).astype(complex)
T = np.diag([1, np.exp(1j * np.pi / 4)])
CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], dtype=complex)   # |control target>
CZ = np.diag([1, 1, 1, -1]).astype(complex)
SWAP = np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]], dtype=complex)

def rx(t): return np.array([[np.cos(t/2), -1j*np.sin(t/2)], [-1j*np.sin(t/2), np.cos(t/2)]])
def ry(t): return np.array([[np.cos(t/2), -np.sin(t/2)], [np.sin(t/2), np.cos(t/2)]], dtype=complex)
def rz(t): return np.diag([np.exp(-1j*t/2), np.exp(1j*t/2)])

def zero_state(n):
    psi = np.zeros(2 ** n, dtype=complex); psi[0] = 1.0
    return psi

## 2. Apply a single-qubit gate
Implement `apply_1q(state, U, q, n)` → new flat state vector. Steps: reshape to `[2]*n`, find the axis of qubit `q` (**`n-1-q`**), `np.tensordot(U, psi, axes=([1], [ax]))`, move the new axis 0 back to `ax` with `np.moveaxis`, flatten.

*Paper check first:* what is $(I\otimes H)\lvert 00\rangle$ in Qiskit order if $H$ acts on qubit 0?

In [ ]:
def apply_1q(state, U, q, n):
    # TODO: reshape, contract U with the axis of qubit q, restore axis order, flatten
    raise NotImplementedError

## 3. Apply a two-qubit gate
`apply_2q(state, U, qa, qb, n)`: `U` is $4\times4$ in the basis $\lvert a\,b\rangle$ with $a$ = qubit `qa` as the most-significant bit **of U**. Reshape `U` to `(2,2,2,2)` = `[a', b', a, b]`, contract axes `[2,3]` with the axes of `qa`, `qb`, then `np.moveaxis(psi, [0, 1], [axa, axb])`.

So `apply_2q(psi, CNOT, c, t, n)` must equal Qiskit's `qc.cx(c, t)`.

In [ ]:
def apply_2q(state, U, qa, qb, n):
    # TODO
    raise NotImplementedError

## 4. Run a circuit
`simulate(ops, n)` starts from $\lvert 0\ldots0\rangle$ and applies a list of ops:
`("h", q)`, `("x", q)`, `("y", q)`, `("z", q)`, `("s", q)`, `("t", q)`, `("rx"|"ry"|"rz", theta, q)`, `("cx"|"cz"|"swap", qa, qb)`.

In [ ]:
def simulate(ops, n):
    # TODO: dispatch each op to apply_1q / apply_2q
    raise NotImplementedError

## 5. Measurement sampling
`measure_counts(state, shots, seed=0)` → dict mapping **Qiskit-style** bitstrings $q_{n-1}\ldots q_0$ (i.e. `format(index, f"0{n}b")`) to counts. Use `np.random.default_rng(seed)` so results are reproducible.

In [ ]:
def measure_counts(state, shots, seed=0):
    # TODO: probabilities |amp|^2, rng.choice over indices, tally into a dict
    raise NotImplementedError

## 6. Experiments
Compare with Qiskit on a GHZ circuit and look at the sampled histogram. Then try a circuit where the ordering matters: `X` on qubit 0 only.

In [ ]:
ops = [("h", 0), ("cx", 0, 1), ("cx", 1, 2)]
mine = simulate(ops, 3)
qc = QuantumCircuit(3); qc.h(0); qc.cx(0, 1); qc.cx(1, 2)
print("mine  :", mine)
print("qiskit:", Statevector(qc).data)
print("match :", np.allclose(mine, Statevector(qc).data))

counts = measure_counts(mine, 2000, seed=1)
plt.bar(list(counts), list(counts.values())); plt.title("GHZ samples (Qiskit bit order q2 q1 q0)"); plt.show()

print("X on qubit 0 of 2 qubits ->", measure_counts(simulate([("x", 0)], 2), 10))

### Reflect (on paper)
1. Why is the reshape approach $O(2^n)$ per gate while the Kronecker approach is $O(4^n)$?
2. Write the textbook (big-endian) vector of the state Qiskit prints as `'01'`. Which permutation maps one convention to the other for $n$ qubits?

## Grade
Run all cells above, then this one. It runs the hidden checks and records the result in the Learning Center.

In [ ]:
from lc_client import grade
grade("qc_01_statevector", globals())